In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from scipy.sparse import csr_matrix

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
samples = list(np.loadtxt(f'{projdir}/txt/samples.txt', dtype=str))
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
s = samples[0]

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/rna/raw/intersect_atac.h5ad')

In [ ]:
adata

In [ ]:
liu = sc.read_h5ad(f'{projdir}//h5ad/rna/liu/seurat_raw.h5ad')

In [ ]:
liu

In [ ]:
liu.obs.columns

In [ ]:
liu.obs['snClust'].value_counts()

In [ ]:
liu.obs['cluster'].value_counts()

In [ ]:
liu.var.index.duplicated()

In [ ]:
liu.X = csr_matrix(liu.X)

In [ ]:
np.mean(liu.X, axis=0), \
np.mean(liu.X, axis=1)

In [ ]:
np.max(liu.X, axis=0).todense(), \
np.max(liu.X, axis=1).todense()

In [ ]:
np.sum(liu.X, axis=0), \
np.sum(liu.X, axis=1)

### start with high-dimensional comparison

In [ ]:
liu.var

In [ ]:
ensg_path = '/u/project/cluo/chliu/Genomes/IGVF_hg38/annotations/ensembl_to_symbol.tsv'
ensg_df = pd.read_csv(ensg_path, sep='\t', header=0, index_col=0)

In [ ]:
ensg_df.head()

In [ ]:
ensg_df['gene_no_colon'] = [x.split('.')[0] for x in ensg_df['gene']]

In [ ]:
ensg_map = dict(zip(ensg_df['gene_no_colon'], ensg_df['symbol']))

In [ ]:
# [ensg_map[x] for x in liu.var.index]

In [ ]:
gene_symbols = []
for x in liu.var.index:
    try:
        gene_symbols.append(ensg_map[x])
    except:
        gene_symbols.append(x)
liu.var['symbol'] = gene_symbols

In [ ]:
n_unmapped = sum(['ENSG' in x for x in liu.var['symbol']])
n_unmapped

In [ ]:
liu.var['symbol'][:5], adata.var.index[:5]

In [ ]:
### lmao kill me
n_unmapped = sum(['ENSG' in x for x in adata.var.index])
n_unmapped

In [ ]:
adata_mask = adata.var.index.isin(liu.var['symbol'])
liu_mask = liu.var['symbol'].isin(adata.var.index)

np.sum(adata_mask), np.sum(liu_mask), adata.var.shape, liu.var.shape

In [ ]:
final_adata = adata[:, adata_mask]
sub_liu = liu[:, liu_mask]

### duplicated for some reason?
sub_liu = sub_liu[:, ~sub_liu.var['symbol'].duplicated()]

In [ ]:
final_adata

In [ ]:
sub_liu

In [ ]:
final_adata.var

In [ ]:
np.sum(final_adata.var.index.isin(sub_liu.var['symbol']))

In [ ]:
np.sum(sub_liu.var['symbol'].isin(final_adata.var.index))

In [ ]:
sub_liu.var.index = sub_liu.var['symbol']

In [ ]:
liu_mtx = pd.DataFrame(sub_liu.X.toarray(), index=sub_liu.obs.index, columns = sub_liu.var.index).transpose()

In [ ]:
liu_mtx.head()

In [ ]:
liu_mtx

In [ ]:
reindexed_liu = csr_matrix(liu_mtx.reindex(final_adata.var.index).values.transpose())
reindexed_var = sub_liu.var.reindex(final_adata.var.index)

In [ ]:
reindexed_liu, reindexed_var

In [ ]:
final_liu = anndata.AnnData(X= reindexed_liu, obs=sub_liu.obs, var = reindexed_var)

In [ ]:
final_liu

In [ ]:
final_liu.write_h5ad(f'{projdir}/h5ad/rna/liu/seurat_reindexed.h5ad')

In [ ]:
final_adata.X

In [ ]:
final_adata.write_h5ad(f'{projdir}/h5ad/rna/liu/igvf_intersect_atac.h5ad')

In [ ]:
sum(final_adata.var.index == final_liu.var['symbol'])

In [ ]:
!date